# Lesson 6.8: Effect Size and Practical Significance

Compute Cohen's d for the memory-training study by hand, then see a huge sample make a tiny effect significant while d stays tiny.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## 1. Enter the two groups

The memory-training study from the lesson: 25 people trained, 25 controls. The scores below are built to match the lesson (trained: mean 78, SD 10; control: mean 70, SD 12).

In [1]:
import numpy as np
from scipy import stats

trained = np.array([95.69, 56.78, 81.95, 73.61, 74.58, 76.59, 61.32, 76.45, 71.09, 106.54, 80.32, 75.43, 76.03, 72.76, 69.48, 75.11, 82.49, 76.39, 86.52, 76.72, 78.62, 91.5, 83.03, 74.14, 76.87])
control = np.array([62.92, 68.16, 88.37, 77.33, 52.05, 70.03, 63.23, 71.72, 52.41, 72.74, 72.67, 87.4, 73.57, 75.7, 53.68, 94.84, 49.03, 82.19, 66.46, 60.41, 62.87, 62.7, 74.26, 68.88, 86.38])

for name, g in [('trained', trained), ('control', control)]:
    print(f'{name}: n = {len(g)}, mean = {g.mean():.2f}, sd = {g.std(ddof=1):.2f}')

trained: n = 25, mean = 78.00, sd = 10.00
control: n = 25, mean = 70.00, sd = 12.00


## 2. Cohen's d by hand

Cohen's d is the gap between the means measured in pooled standard deviations. The pooled SD is a weighted average of the two group SDs, so d does not depend on the sample size.

In [2]:
n1, n2 = len(trained), len(control)
s1, s2 = trained.std(ddof=1), control.std(ddof=1)

sp = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
d = (trained.mean() - control.mean()) / sp
g = d * (1 - 3 / (4 * (n1 + n2 - 2) - 1))   # Hedges' g, a small-sample correction

print(f'pooled SD = {sp:.3f}')
print(f"Cohen's d = {d:.3f}")
print(f"Hedges' g = {g:.3f}")

pooled SD = 11.045
Cohen's d = 0.724
Hedges' g = 0.713


## 3. Add the p-value

The t-test says whether the gap could be chance, while d says how big it is. Report both. (This is the pooled test; the lesson's Welch version gives the same p to three decimals.)

In [3]:
res = stats.ttest_ind(trained, control)

print(f't = {res.statistic:.3f}, df = {res.df:.0f}, p = {res.pvalue:.4f}')

t = 2.561, df = 48, p = 0.0136


## 4. Tiny effect, huge sample

ttest_ind_from_stats runs the t-test from summary numbers. Compare three studies: a huge sample with a trivial gap, the memory study, and a tiny sample with a big gap. The p-values and d tell different stories.

In [4]:
def study(label, diff, sd, n):
    res = stats.ttest_ind_from_stats(diff, sd, n, 0, sd, n)   # pooled test
    print(f'{label:<14} d = {diff / sd:.3f}  t = {res.statistic:.2f}  p = {res.pvalue:.4f}')

study('huge sample', 0.2, 15, 50000)
study('memory study', 8, sp, 25)
study('tiny sample', 15, 15, 4)

huge sample    d = 0.013  t = 2.11  p = 0.0350
memory study   d = 0.724  t = 2.56  p = 0.0136
tiny sample    d = 1.000  t = 1.41  p = 0.2070


## Try this

- Change the huge sample size from 50,000 to 500 and predict what happens to p and to d.
- In the tiny-sample study, raise n from 4 to 40 and predict which of d and p changes, then run it.

## The same idea in R, SPSS and Stata

**R**

```r
trained <- c(95.69, 56.78, 81.95, 73.61, 74.58, 76.59, 61.32, 76.45, 71.09, 106.54, 80.32, 75.43, 76.03, 72.76, 69.48, 75.11, 82.49, 76.39, 86.52, 76.72, 78.62, 91.5, 83.03, 74.14, 76.87)
control <- c(62.92, 68.16, 88.37, 77.33, 52.05, 70.03, 63.23, 71.72, 52.41, 72.74, 72.67, 87.4, 73.57, 75.7, 53.68, 94.84, 49.03, 82.19, 66.46, 60.41, 62.87, 62.7, 74.26, 68.88, 86.38)
sp <- sqrt((24 * var(trained) + 24 * var(control)) / 48)
(mean(trained) - mean(control)) / sp          # Cohen's d
t.test(trained, control, var.equal = TRUE)

study <- function(diff, sd, n) {
  t <- diff / sqrt(2 * sd^2 / n)
  c(d = diff / sd, t = t, p = 2 * pt(-abs(t), 2 * n - 2))
}
study(0.2, 15, 50000)
study(15, 15, 4)
```

**SPSS** (Analyze > Compare Means > Independent-Samples T Test (version 27 and later also prints an effect-size table with Cohen's d))

```
DATA LIST FREE / label n diff sd.
BEGIN DATA
1 50000 0.2 15
2 25 8 11.0454
3 4 15 15
END DATA.
COMPUTE d = diff / sd.
COMPUTE t = diff / SQRT(2 * sd**2 / n).
COMPUTE p = 2 * CDF.T(-ABS(t), 2 * n - 2).
LIST label d t p.
```

**Stata**

```
esizei 25 78 10 25 70 12, cohensd      // Cohen's d from summary numbers
ttesti 25 78 10 25 70 12
esizei 50000 0.2 15 50000 0 15, cohensd
ttesti 50000 0.2 15 50000 0 15
```